In [1]:
import os
import sentencepiece as spm

from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os
print(os.getcwd())

/content


In [4]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Indic-Multimodal-NMT"
)

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)

/content/drive/MyDrive/Indic-Multimodal-NMT


In [5]:
!pip install datasets==2.19.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 10.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


In [6]:
print(f"Does '{PROJECT_ROOT}' exist? {PROJECT_ROOT.exists()}")

if PROJECT_ROOT.exists():
    print(f"Contents of '{PROJECT_ROOT}':")
    for item in PROJECT_ROOT.iterdir():
        print(item.name)
else:
    print(f"The folder '{PROJECT_ROOT}' does not exist. Please ensure Google Drive is mounted correctly and the path is accurate.")

Does '/content/drive/MyDrive/Indic-Multimodal-NMT' exist? True
Contents of '/content/drive/MyDrive/Indic-Multimodal-NMT':
data
Untitled1.ipynb
Dataset Exploration.ipynb


In [1]:
from datasets import load_dataset

flickr = load_dataset(
    "nlphuji/flickr30k",
    split="test"
)

Generating test split:   0%|          | 0/31014 [00:00<?, ? examples/s]

In [ ]:
flickr

In [ ]:
import os
from pathlib import Path

IMAGE_DIR = Path(
    "data/raw/flickr30k/images"
)
os.makedirs(PROJECT_ROOT + "/" + IMAGE_DIR, exist_ok=True)
print(f"Created directory: {IMAGE_DIR}")

In [ ]:
for idx, item in enumerate(flickr):

    filename = item["filename"]

    image_path = IMAGE_DIR / filename

    if not image_path.exists():
        item["image"].save(image_path)


    if idx % 1000 == 0:
        print(
            f"Saved {idx} images"
        )

In [ ]:
sample = flickr[0]

sample.keys()

In [ ]:
print(sample["caption"])
print(sample["filename"])
print(sample["img_id"])

In [ ]:
from IPython.display import display

display(sample["image"])

print("Caption:")
print(sample["caption"])

In [ ]:
records = []

for item in flickr:

    for caption in item["caption"]:

        records.append(
            {
                "image_id": item["img_id"],
                "filename": item["filename"],
                "source_text": caption
            }
        )


df = pd.DataFrame(records)


print("Total image-caption pairs:", len(df))
print("Unique images:", df["filename"].nunique())

In [ ]:
df.shape

In [ ]:
PROCESSED_DIR = Path(
    "data/processed/"
)
os.makedirs(PROCESSED_DIR, exist_ok=True)
print(f"Created directory: {PROCESSED_DIR}")

In [ ]:
df.to_csv(
    "data/processed/flickr30k_metadata.csv",
    index=False
)

In [ ]:
from IPython.display import display

display(sample["image"])

print(sample["caption"])
print(sample["filename"])

In [ ]:
import random

idx = random.randint(0, len(df)-1)

sample = df.iloc[idx]

sample

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

row = df.iloc[5135]

image_path = Path(IMAGE_DIR) / row["filename"]

image = Image.open(image_path)


plt.figure(figsize=(6,6))
plt.imshow(image)
plt.axis("off")
plt.show()


print("Captions:")
for i, caption in enumerate(row["source_text"]):
    print(i+1, ":", caption)

In [ ]:
print("Total image-caption pairs:", len(df))

print(
    "Unique images:",
    df["filename"].nunique()
)
sample = flickr[0]

print(sample["filename"])
print(type(sample["caption"]))
print(sample["caption"])

In [ ]:
IMAGE_DIR.exists()

In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt


for _ in range(5):

    sample = df.sample(1).iloc[0]

    image_path = (
        IMAGE_DIR /
        sample["filename"]
    )

    image = Image.open(image_path)


    plt.figure(figsize=(5,5))

    plt.imshow(image)

    plt.axis("off")

    plt.title(
        sample["source_text"],
        fontsize=10
    )

    plt.show()

In [ ]:
print(
    "Image-caption pairs:",
    len(df)
)

print(
    "Unique images:",
    df["filename"].nunique()
)

In [ ]:
df["caption_length"] = (
    df["source_text"]
    .apply(
        lambda x: len(x.split())
    )
)

In [ ]:
df["caption_length"].describe()

This helps decide:

- max sequence length
- tokenizer vocabulary size

In [ ]:
from collections import Counter


words = []

for caption in df["source_text"]:

    words.extend(
        caption.lower().split()
    )


counter = Counter(words)


counter.most_common(20)

4.3 Most common words

Useful for understanding the corpus.

In [ ]:
df = df[
    [
        "image_id",
        "filename",
        "source_text"
    ]
]

In [ ]:
df.to_csv(
    "data/processed/flickr30k_metadata.csv",
    index=False
)